# Suivre, enregistrer et rejouer un entraînement Spark avec MLflow

Ce TP suit avec MLflow l'entraînement d'un classifieur Spark ML qui prédit si un avis sur un médicament est positif. Vous notez à la main les paramètres, les métriques, les données et les artefacts d'une exécution, comparez plusieurs exécutions et leurs courbes de perte, regardez ce que l'autologging de Spark ML note d'une validation croisée, enregistrez le modèle au format MLflow et le rechargez. Vous faites ensuite de l'entraînement un projet MLflow à deux points d'entrée, que vous lancez avec `mlflow run`, puis promouvez son meilleur modèle dans le registre et l'évaluez par son alias.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab : un entraînement prend une demi-minute environ, la validation croisée trois minutes, une exécution du projet deux minutes, et le notebook entier un quart d'heure. Le modèle lui-même reste simple et n'est pas le sujet : le TP d'apprentissage avec Spark ML le travaille en détail.

## Installation et données

Les données sont le [Drug Review Dataset](https://archive.ics.uci.edu/dataset/462/drug+review+dataset+drugs+com) (Gräßer et Kallumadi, UCI Machine Learning Repository, 2018, licence CC BY 4.0) : des avis publiés sur Drugs.com de 2008 à 2017, en anglais. Ce TP n'en prend que la partie test, `drugsComTest_raw.tsv` (28 Mo, 53 766 avis), pour que les entraînements restent courts. Chaque ligne est un avis : le médicament (`drugName`), l'affection soignée (`condition`), le texte (`review`), la note de 1 à 10 (`rating`), la date (`date`) et le nombre de lecteurs qui ont trouvé l'avis utile (`usefulCount`).

In [ ]:
!pip install pyspark mlflow
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv

In [ ]:
import html
import os
import sys
import time
from pathlib import Path

import mlflow
import pandas as pd
import pyspark
from mlflow.models import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import IDF, CountVectorizer, RegexTokenizer, StopWordsRemover
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MLflow range les exécutions dans un *tracking store* que désigne la variable d'environnement `MLFLOW_TRACKING_URI` : ici une base SQLite, `mlflow.db`, dans le dossier du notebook, et les artefacts (fichiers, modèles) dans son sous-dossier `mlruns/`. Le chemin est absolu pour que les processus que lance `mlflow run`, à l'exercice 6, écrivent dans la même base : ils héritent de la variable. Sur Colab, ce dossier disparaît avec la session ; hors de Colab, `mlflow ui --backend-store-uri sqlite:///mlflow.db`, lancé dans ce dossier, ouvre l'interface de MLflow sur http://localhost:5000.

La session Spark tourne en local sur deux cœurs, avec quatre partitions pour les agrégations au lieu des 200 par défaut. Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance prennent l'interpréteur de ce notebook, `sys.executable` : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut être d'une autre version de Python que celui du driver, ce que Spark refuse.

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{Path.cwd() / 'mlflow.db'}"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("avis-mlflow")
  .master("local[2]")
  .config("spark.driver.memory", "3g")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(mlflow.__version__, pyspark.__version__, mlflow.get_tracking_uri())

## Les données et le modèle, déjà écrits

Ce code lit les avis (un avis peut tenir sur plusieurs lignes), retire les doublons (un même avis publié sous le nom commercial et sous le nom de la molécule), décode le texte et ajoute l'étiquette `positif`, 1 pour une note de 7 ou plus. Il coupe les avis en entraînement (80 %) et test (20 %) d'après leur texte : un avis va au test quand l'empreinte `xxhash64` de son texte, en valeur absolue, est un multiple de 5. Le découpage ne dépend ainsi que des données : le même à chaque lecture, que les deux parties soient en cache ou non, et le même dans le script du projet MLflow, plus loin. Elles sont mises en cache, puisque chaque entraînement les relit.

`pipeline_avis(reg, vocabulaire)` rend un pipeline neuf de cinq étapes, nommées comme dans le cours : `mots` coupe l'avis en mots, `utiles` retire les mots vides, `frequences` compte les `vocabulaire` mots les plus fréquents, `tfidf` les pondère, et `reglog` est une régression logistique de paramètre de régularisation `reg`. `evaluateur` calcule l'aire sous la courbe ROC (AUC) d'un DataFrame de prédictions.

In [ ]:
decoder = F.udf(lambda texte: html.unescape(texte[1:-1]), "string")


def lire_avis(fichier):
  return (
    spark.read.csv(fichier, sep="\t", header=True, multiLine=True, escape='"')
    .dropDuplicates(["review", "date", "rating"])
    .select(
      decoder("review").alias("review"),
      (F.col("rating").cast("double") >= 7).cast("double").alias("positif"),
    )
  )


def pipeline_avis(reg, vocabulaire):
  mots = RegexTokenizer(inputCol="review", outputCol="mots", pattern=r"[^\w']+")
  utiles = StopWordsRemover(inputCol="mots", outputCol="mots_utiles")
  frequences = CountVectorizer(
    inputCol="mots_utiles", outputCol="tf", vocabSize=vocabulaire, minDF=5
  )
  tfidf = IDF(inputCol="tf", outputCol="features")
  reglog = LogisticRegression(labelCol="positif", regParam=reg, maxIter=50)
  return Pipeline(stages=[mots, utiles, frequences, tfidf, reglog])


avis = lire_avis("drugsComTest_raw.tsv")
au_test = F.abs(F.xxhash64("review")) % 5 == 0
entrainement, test = avis.where(~au_test).cache(), avis.where(au_test).cache()
print(entrainement.count(), test.count())
evaluateur = BinaryClassificationEvaluator(labelCol="positif")

Trois avis écrits pour l'occasion, pour essayer les modèles enregistrés :

In [ ]:
nouveaux_avis = pd.DataFrame(
  {
    "review": [
      "This medication changed my life, my anxiety is finally under control.",
      "Terrible side effects: constant nausea and headaches. I stopped after a week.",
      "It helped a little at first, then nothing. Not worth the price.",
    ]
  }
)

## 1. Une exécution suivie à la main

Choisissez l'expérience `avis-medicaments` (`mlflow.set_experiment` la crée si elle n'existe pas). Dans une exécution nommée `premiere`, notez :

- une balise : `modele` vaut `tfidf-reglog` ;
- deux paramètres : `reg` vaut 0,1 et `vocabulaire` vaut 5 000 ;
- les données d'entraînement : un jeu de données MLflow tiré de `entrainement` (`mlflow.data.from_spark`), nommé `entrainement`, avec pour chemin le fichier lu, `drugsComTest_raw.tsv`, rattaché à l'exécution dans le contexte `training` ;
- deux métriques : la durée de l'entraînement de `pipeline_avis(0.1, 5000)` sur `entrainement`, en secondes (`duree_s`), et l'AUC du modèle sur `test` (`auc_test`) ;
- un artefact : les 20 premiers mots du vocabulaire qu'a appris l'étape `frequences` (la troisième du modèle), dans un dictionnaire JSON, `vocabulaire.json`.

Gardez l'identifiant de l'exécution dans `id_premiere`. Relisez-la ensuite avec `mlflow.get_run` et affichez ses paramètres, ses métriques et ses balises, puis ses données (nom, type de source, source, schéma, profil), la liste de ses artefacts et le contenu de `vocabulaire.json`. Quel est le type des paramètres relus ? Quelles balises MLflow a-t-il posées lui-même ? Que garde le jeu de données noté : les avis, ou une description ?


**Documentation utile**

- [mlflow.data.from_spark](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.data.html#mlflow.data.from_spark)
- [mlflow.log_dict](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.log_dict)
- [mlflow.get_run](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.get_run)
- [mlflow.client.MlflowClient.list_artifacts](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.list_artifacts)
- [mlflow.artifacts.load_dict](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.artifacts.html#mlflow.artifacts.load_dict)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`mlflow.start_run` s'utilise dans un `with` : l'exécution se termine à la sortie du bloc, même sur une erreur (statut `FAILED`). Son objet `info.run_id` donne l'identifiant. `mlflow.data.from_spark(df, path=..., name=...)` décrit un DataFrame Spark, que `mlflow.log_input(donnees, context=...)` rattache à l'exécution. Le vocabulaire appris est la liste `vocabulary` de `modele.stages[2]` ; `mlflow.log_dict(dictionnaire, "vocabulaire.json")` l'écrit en artefact. L'exécution relue a les dictionnaires `data.params`, `data.metrics` et `data.tags`, et la liste `inputs.dataset_inputs`, dont chaque élément a un `dataset` (attributs `name`, `source_type`, `source`, `schema`, `profile`). `mlflow.MlflowClient().list_artifacts(id)` liste les artefacts d'une exécution, `mlflow.artifacts.load_dict` en relit un.

```python
mlflow.set_experiment(...)
with mlflow.start_run(run_name=...) as execution:
  mlflow.set_tag(...)
  mlflow.log_params({...})
  donnees = mlflow.data.from_spark(entrainement, path=..., name=...)
  mlflow.log_input(donnees, context=...)
  debut = time.time()
  modele = pipeline_avis(...).fit(...)
  mlflow.log_metric("duree_s", time.time() - debut)
  mlflow.log_metric("auc_test", evaluateur.evaluate(...))
  mlflow.log_dict({"mots": ...}, "vocabulaire.json")
id_premiere = execution.info.run_id
relue = mlflow.get_run(id_premiere)
print(relue.data.params)
for entree in relue.inputs.dataset_inputs:
  print(entree.dataset.name, ...)
print([a.path for a in mlflow.MlflowClient().list_artifacts(...)])
print(mlflow.artifacts.load_dict(f"runs:/{id_premiere}/vocabulaire.json"))
```

### Solution

In [ ]:
mlflow.set_experiment("avis-medicaments")
with mlflow.start_run(run_name="premiere") as execution:
  mlflow.set_tag("modele", "tfidf-reglog")
  mlflow.log_params({"reg": 0.1, "vocabulaire": 5000})
  donnees = mlflow.data.from_spark(
    entrainement, path="drugsComTest_raw.tsv", name="entrainement"
  )
  mlflow.log_input(donnees, context="training")
  debut = time.time()
  modele = pipeline_avis(0.1, 5000).fit(entrainement)
  mlflow.log_metric("duree_s", time.time() - debut)
  mlflow.log_metric("auc_test", evaluateur.evaluate(modele.transform(test)))
  mlflow.log_dict({"mots": modele.stages[2].vocabulary[:20]}, "vocabulaire.json")
id_premiere = execution.info.run_id
relue = mlflow.get_run(id_premiere)
print(relue.data.params)
print(relue.data.metrics)
print(relue.data.tags)
for entree in relue.inputs.dataset_inputs:
  print(entree.dataset.name, entree.dataset.source_type, entree.dataset.source)
  print(entree.dataset.schema, entree.dataset.profile)
print([a.path for a in mlflow.MlflowClient().list_artifacts(id_premiere)])
print(mlflow.artifacts.load_dict(f"runs:/{id_premiere}/vocabulaire.json"))
# Les paramètres reviennent en chaînes ({'reg': '0.1', 'vocabulaire': '5000'}) :
# MLflow les garde comme du texte. AUC de test de 0,863, en une demi-minute
# environ. MLflow a posé lui-même mlflow.user, mlflow.source.name et
# mlflow.source.type (NOTEBOOK, le programme qui a lancé le code) et mlflow.runName.
# Le jeu de données noté n'est qu'une description : son nom, sa source (spark, le
# chemin du fichier), son schéma (review, une chaîne, et positif, un nombre), un
# profil (approx_count, un nombre d'avis estimé : 38 818 pour 38 682) et une
# empreinte (digest), calculée sur le plan de la requête, pas sur les avis. MLflow
# ne copie pas les données : le fichier doit rester là où le chemin le dit. Le
# vocabulaire commence par day, side, taking, years, effects : les mots les plus
# fréquents des avis, une fois les mots vides retirés.

## 2. Comparer des exécutions

Entraînez `pipeline_avis(reg, 5000)` pour `reg` valant 0,01, 0,3 et 1, chacun dans son exécution, nommée `reg-<valeur>`, avec la même balise, les mêmes paramètres et les mêmes métriques qu'à l'exercice 1 (sans les données ni le vocabulaire). Notez en plus la perte de la régression logistique à chaque itération, en métrique `perte` avec le numéro de l'itération pour pas (`step`) : le résumé du modèle, dernière étape du pipeline, la donne.

Cherchez ensuite les exécutions de l'expérience, triées par AUC décroissante, et affichez leur nom, `reg` et `auc_test`. Quelle régularisation garder ?

Relisez aussi, pour chacune des trois exécutions `reg-…`, sa série `perte` avec `MlflowClient().get_metric_history`, et affichez son dernier pas (le nombre d'itérations), sa première et sa dernière valeur. Quel `reg` converge en le moins d'itérations ? La convergence la plus rapide donne-t-elle la meilleure AUC ? Les pertes finales se comparent-elles d'un `reg` à l'autre ?

Rouvrez enfin l'exécution `premiere` et essayez d'y changer `reg` en 0,5 : que se passe-t-il ? Posez-y plutôt une balise `retenu` qui vaut `non`. Pourquoi MLflow traite-t-il les paramètres et les balises différemment ?


**Documentation utile**

- [mlflow.client.MlflowClient.get_metric_history](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.get_metric_history)
- [mlflow.set_tag](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.set_tag)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`modele.stages[-1]` est la régression logistique entraînée ; son `summary.objectiveHistory` est la liste des pertes. `mlflow.search_runs` rend un DataFrame pandas, une colonne par paramètre (`params.reg`), métrique (`metrics.auc_test`) et balise (`tags.mlflow.runName`). `mlflow.MlflowClient().get_metric_history(id, "perte")` rend la série d'une métrique, une liste d'objets aux attributs `step` et `value`. `mlflow.start_run(run_id=...)` rouvre une exécution terminée. Un changement de paramètre lève une `mlflow.exceptions.MlflowException`.

```python
ids_reg = {}
for reg in [0.01, 0.3, 1.0]:
  with mlflow.start_run(run_name=f"reg-{reg}") as execution:
    ...
    for etape, perte in enumerate(modele.stages[-1].summary.objectiveHistory):
      mlflow.log_metric("perte", perte, step=etape)
  ids_reg[reg] = execution.info.run_id
executions = mlflow.search_runs(
  experiment_names=[...], order_by=["metrics.auc_test DESC"]
)
print(executions[["tags.mlflow.runName", "params.reg", "metrics.auc_test"]])
client = mlflow.MlflowClient()
for reg, id_reg in ids_reg.items():
  historique = client.get_metric_history(id_reg, "perte")
  print(reg, historique[-1].step, ...)
with mlflow.start_run(run_id=id_premiere):
  try:
    mlflow.log_param("reg", 0.5)
  except mlflow.exceptions.MlflowException as erreur:
    print(erreur)
  mlflow.set_tag(...)
```

### Solution

In [ ]:
ids_reg = {}
for reg in [0.01, 0.3, 1.0]:
  with mlflow.start_run(run_name=f"reg-{reg}") as execution:
    mlflow.set_tag("modele", "tfidf-reglog")
    mlflow.log_params({"reg": reg, "vocabulaire": 5000})
    debut = time.time()
    modele = pipeline_avis(reg, 5000).fit(entrainement)
    mlflow.log_metric("duree_s", time.time() - debut)
    mlflow.log_metric("auc_test", evaluateur.evaluate(modele.transform(test)))
    for etape, perte in enumerate(modele.stages[-1].summary.objectiveHistory):
      mlflow.log_metric("perte", perte, step=etape)
  ids_reg[reg] = execution.info.run_id
executions = mlflow.search_runs(
  experiment_names=["avis-medicaments"], order_by=["metrics.auc_test DESC"]
)
print(executions[["tags.mlflow.runName", "params.reg", "metrics.auc_test"]])
client = mlflow.MlflowClient()
for reg, id_reg in ids_reg.items():
  historique = client.get_metric_history(id_reg, "perte")
  print(
    reg,
    historique[-1].step,
    round(historique[0].value, 3),
    round(historique[-1].value, 3),
  )
with mlflow.start_run(run_id=id_premiere):
  try:
    mlflow.log_param("reg", 0.5)
  except mlflow.exceptions.MlflowException as erreur:
    print(str(erreur).splitlines()[0])
  mlflow.set_tag("retenu", "non")
# AUC de 0,865 pour reg = 0,3, 0,863 pour 0,1, 0,861 pour 1 et 0,847 pour 0,01 : on
# garde 0,3, mais les écarts sont faibles au-delà de 0,1. reg = 1 converge en 13
# itérations, 0,3 en 18 et 0,01 en 41 (aucun n'atteint maxIter = 50) : plus la
# régularisation est forte, plus le problème est facile à optimiser. La convergence la
# plus rapide ne donne pas la meilleure AUC : elle dit ce qu'a coûté l'optimisation,
# pas ce que vaut le modèle. Les pertes partent toutes de 0,640, celle du modèle aux
# coefficients nuls, qui ne connaît que la proportion de positifs ; leurs valeurs
# finales (0,352 pour 0,01, 0,498 pour 0,3, 0,563 pour 1) ne se comparent pas : la
# perte comprend la pénalité, qui croît avec reg. Changer un paramètre lève une
# MlflowException (« Changing param values is not allowed ») : un paramètre décrit ce
# qui a produit les métriques et le modèle de l'exécution, et le changer après coup
# les rendrait faux. Une balise est une annotation (un statut, un commentaire) : on
# peut la poser et la changer à tout moment.

## 3. L'autologging de Spark ML

La cellule d'exercice prépare une validation croisée à deux plis (`CrossValidator`), qui choisit `reg` parmi 0,1 et 0,3 pour `pipeline_avis(0.1, 5000)` d'après `evaluateur` ; elle ne lance encore rien. Activez l'autologging de Spark ML, puis, dans une exécution nommée `validation-croisee`, ajustez la validation croisée sur `entrainement` et calculez l'AUC du meilleur modèle sur `test`, toujours dans l'exécution, en appelant `evaluateur.evaluate`. Désactivez enfin l'autologging : les exercices suivants notent eux-mêmes ce qu'ils veulent.

Vous n'avez noté vous-même ni paramètre, ni métrique, ni artefact. Qu'a noté l'autologging ? Affichez :

- les exécutions enfants de `validation-croisee` (la balise `mlflow.parentRunId` porte l'identifiant de leur parent), avec leur `reg` et leur AUC moyenne ;
- les paramètres et les métriques de l'exécution parent, et son artefact `best_parameters.json` ;
- la liste de ses artefacts, et ses données, comme à l'exercice 1.

Combien d'enfants a-t-elle, et combien d'entraînements la validation croisée a-t-elle faits ? Quel nom l'AUC sur `test` a-t-elle reçu, et d'où vient-il ? Affichez aussi `entrainement.is_cached` avant et après la validation croisée : qu'a fait l'autologging du cache ?


**Documentation utile**

- [mlflow.artifacts.load_dict](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.artifacts.html#mlflow.artifacts.load_dict)
- [mlflow.client.MlflowClient.list_artifacts](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.list_artifacts)

In [ ]:
pipeline = pipeline_avis(0.1, 5000)
reglog = pipeline.getStages()[-1]
grille = ParamGridBuilder().addGrid(reglog.regParam, [0.1, 0.3]).build()
validation = CrossValidator(
  estimator=pipeline,
  estimatorParamMaps=grille,
  evaluator=evaluateur,
  numFolds=2,
  parallelism=2,
  seed=42,  # sans graine fixe, les plis changeraient à chaque noyau
)
# Votre code ici

### Conseils supplémentaires

`mlflow.pyspark.ml.autolog()` active l'autologging, `disable=True` le désactive. Dans les exécutions enfants, le paramètre s'appelle `params.LogisticRegression.regParam` et la métrique `metrics.avg_areaUnderROC`. `mlflow.artifacts.load_dict` lit un artefact JSON, `mlflow.MlflowClient().list_artifacts` liste ceux d'une exécution. Un DataFrame sorti du cache y retourne par `cache()`.

```python
print(entrainement.is_cached)
mlflow.pyspark.ml.autolog()
with mlflow.start_run(run_name=...) as execution:
  modele_cv = validation.fit(...)
  evaluateur.evaluate(...)
mlflow.pyspark.ml.autolog(disable=True)
print(entrainement.is_cached)
id_cv = execution.info.run_id
enfants = mlflow.search_runs(
  experiment_names=[...], filter_string=f"tags.mlflow.parentRunId = '{id_cv}'"
)
parent = mlflow.get_run(id_cv)
print(parent.data.params)
print(mlflow.artifacts.load_dict(f"runs:/{id_cv}/best_parameters.json"))
print([a.path for a in mlflow.MlflowClient().list_artifacts(id_cv)])
for entree in parent.inputs.dataset_inputs:
  print(entree.dataset.name, [t.value for t in entree.tags])
```

### Solution

In [ ]:
print(entrainement.is_cached)
mlflow.pyspark.ml.autolog()
with mlflow.start_run(run_name="validation-croisee") as execution:
  modele_cv = validation.fit(entrainement)
  evaluateur.evaluate(modele_cv.transform(test))
mlflow.pyspark.ml.autolog(disable=True)
print(entrainement.is_cached)
entrainement.cache()
id_cv = execution.info.run_id
enfants = mlflow.search_runs(
  experiment_names=["avis-medicaments"],
  filter_string=f"tags.mlflow.parentRunId = '{id_cv}'",
)
print(enfants[["params.LogisticRegression.regParam", "metrics.avg_areaUnderROC"]])
parent = mlflow.get_run(id_cv)
print(parent.data.params)
print(parent.data.metrics)
print(mlflow.artifacts.load_dict(f"runs:/{id_cv}/best_parameters.json"))
print([a.path for a in mlflow.MlflowClient().list_artifacts(id_cv)])
for entree in parent.inputs.dataset_inputs:
  print(entree.dataset.name, [t.value for t in entree.tags])
# Deux exécutions enfants, une par valeur de reg (et non par pli), avec tous les
# paramètres du pipeline et l'AUC moyenne sur les deux plis (0,843 pour 0,3, 0,836
# pour 0,1), et son écart type. Cinq entraînements en tout : deux valeurs sur deux
# plis, puis la meilleure réentraînée sur tout entrainement. Le parent garde les
# réglages de la validation croisée (numFolds, seed…) et de l'évaluateur, le meilleur
# paramètre (best_LogisticRegression.regParam, et best_parameters.json : reg = 0,3),
# le modèle (model, best_model), le tableau de la recherche (search_results.csv), et
# deux jeux de données, l'un d'entraînement (contexte train), l'autre d'évaluation
# (eval). L'AUC sur test s'appelle areaUnderROC_test (0,865) : MLflow nomme une
# métrique calculée après l'entraînement d'après la métrique de l'évaluateur et le nom
# de la variable du DataFrame évalué, test. Piège : is_cached passe de True à False.
# L'autologging met le DataFrame d'entraînement en cache le temps du fit, puis le
# retire du cache, même s'il y était avant : sans le cache() remis, chaque
# entraînement suivant relirait et décoderait le fichier. Le découpage, lui, ne change
# pas : il ne dépend que du texte des avis.

## 4. Enregistrer le modèle et le recharger

Dans une exécution nommée `modele`, entraînez `pipeline_avis(0.3, 5000)` sur `entrainement`, notez `reg`, `vocabulaire` et `auc_test`, puis enregistrez le modèle au format MLflow, avec la saveur `spark`, sous le chemin d'artefact `modele`. Donnez-lui une signature, déduite des deux premiers avis de `nouveaux_avis` (la colonne `review`) et des prédictions du modèle sur eux, et ces deux avis pour exemple d'entrée. Gardez l'identifiant de l'exécution dans `id_modele`.

Affichez le fichier `MLmodel` du modèle enregistré. Quelles saveurs a-t-il ? Rechargez ensuite le modèle de deux façons :

- avec la saveur `spark`, et calculez `probability` et `prediction` des trois avis de `nouveaux_avis`, mis dans un DataFrame Spark ;
- avec la saveur `python_function`, et prédisez directement sur `nouveaux_avis`, un DataFrame pandas.

Les deux donnent-elles les mêmes prédictions ?


**Documentation utile**

- [mlflow.artifacts.download_artifacts](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.artifacts.html#mlflow.artifacts.download_artifacts)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`infer_signature(entree, sortie)` déduit la signature de deux DataFrames pandas : l'entrée (`nouveaux_avis.head(2)`) et la sortie (la colonne `prediction` du modèle, convertie avec `toPandas()`). `mlflow.spark.log_model` prend le modèle et `artifact_path` ; il rend un objet dont `model_uri` est l'URI du modèle, `runs:/<id>/modele`. `mlflow.artifacts.download_artifacts` copie un artefact en local et rend son chemin.

```python
with mlflow.start_run(run_name="modele") as execution:
  ...
  exemple = nouveaux_avis.head(2)
  sortie = (
    modele.transform(spark.createDataFrame(exemple)).select("prediction").toPandas()
  )
  infos = mlflow.spark.log_model(
    modele, artifact_path=..., signature=..., input_example=...
  )
id_modele = execution.info.run_id
dossier = mlflow.artifacts.download_artifacts(f"{infos.model_uri}/MLmodel")
print(Path(dossier).read_text())
modele_spark = mlflow.spark.load_model(infos.model_uri)
modele_pyfunc = mlflow.pyfunc.load_model(infos.model_uri)
```

### Solution

In [ ]:
with mlflow.start_run(run_name="modele") as execution:
  mlflow.log_params({"reg": 0.3, "vocabulaire": 5000})
  modele = pipeline_avis(0.3, 5000).fit(entrainement)
  mlflow.log_metric("auc_test", evaluateur.evaluate(modele.transform(test)))
  exemple = nouveaux_avis.head(2)
  sortie = (
    modele.transform(spark.createDataFrame(exemple)).select("prediction").toPandas()
  )
  infos = mlflow.spark.log_model(
    modele,
    artifact_path="modele",
    signature=infer_signature(exemple, sortie),
    input_example=exemple,
  )
id_modele = execution.info.run_id
print(
  Path(mlflow.artifacts.download_artifacts(f"{infos.model_uri}/MLmodel")).read_text()
)
modele_spark = mlflow.spark.load_model(infos.model_uri)
modele_spark.transform(spark.createDataFrame(nouveaux_avis)).select(
  "probability", "prediction"
).show(truncate=False)
modele_pyfunc = mlflow.pyfunc.load_model(infos.model_uri)
print(modele_pyfunc.predict(nouveaux_avis))
# Deux saveurs : python_function (chargée par le module mlflow.spark) et spark (un
# PipelineModel de PySpark 4.3.0). Le fichier note aussi la signature (une colonne
# review, une chaîne, en entrée ; une colonne prediction en sortie) et l'exemple
# d'entrée. Les deux chargements donnent les mêmes prédictions, 1, 0 et 1 : le
# deuxième avis n'est négatif que de peu (probabilité d'être positif de 0,49), et le
# troisième, négatif, passe pour positif (0,70) ; le modèle reste simple.

## Le script d'entraînement du projet

Un projet MLflow est un dossier (ou un dépôt git) qui contient le code d'un entraînement et un fichier `MLproject`, qui dit comment le lancer. Cette cellule crée le dossier `projet-avis` et y écrit le code : `avis.py` crée la session Spark (`session`), lit et découpe les avis comme ce notebook (`lire_decouper`, qui rend `entrainement` et `test`) ; `entrainer.py` lit le fichier des avis, `reg` et `vocabulaire` sur sa ligne de commande, et entraîne et suit le pipeline comme les exercices précédents, les données notées d'après le fichier reçu. Son `mlflow.start_run()` n'ouvre pas une exécution neuve : lancé par `mlflow run`, il rejoint celle que `mlflow run` a créée, dont il reçoit l'identifiant par la variable d'environnement `MLFLOW_RUN_ID`.

In [ ]:
Path("projet-avis").mkdir(exist_ok=True)
Path("projet-avis/avis.py").write_text(
  """
import html

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def session(nom):
  spark = (
    SparkSession.builder.appName(nom)
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
  )
  spark.sparkContext.setLogLevel("ERROR")
  return spark


def lire_decouper(spark, fichier):
  decoder = F.udf(lambda texte: html.unescape(texte[1:-1]), "string")
  avis = (
    spark.read.csv(fichier, sep="\\t", header=True, multiLine=True, escape='"')
    .dropDuplicates(["review", "date", "rating"])
    .select(
      decoder("review").alias("review"),
      (F.col("rating").cast("double") >= 7).cast("double").alias("positif"),
    )
  )
  au_test = F.abs(F.xxhash64("review")) % 5 == 0
  return avis.where(~au_test).cache(), avis.where(au_test).cache()
""".lstrip()
)
Path("projet-avis/entrainer.py").write_text(
  """
import argparse

import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import IDF, CountVectorizer, RegexTokenizer, StopWordsRemover

from avis import lire_decouper, session

arguments = argparse.ArgumentParser()
arguments.add_argument("donnees")
arguments.add_argument("--reg", type=float, default=0.1)
arguments.add_argument("--vocabulaire", type=float, default=5000)
options = arguments.parse_args()

spark = session("projet-avis")
entrainement, test = lire_decouper(spark, options.donnees)
mots = RegexTokenizer(inputCol="review", outputCol="mots", pattern=r"[^\\w']+")
utiles = StopWordsRemover(inputCol="mots", outputCol="mots_utiles")
frequences = CountVectorizer(
  inputCol="mots_utiles", outputCol="tf", vocabSize=int(options.vocabulaire), minDF=5
)
tfidf = IDF(inputCol="tf", outputCol="features")
reglog = LogisticRegression(labelCol="positif", regParam=options.reg, maxIter=50)
pipeline = Pipeline(stages=[mots, utiles, frequences, tfidf, reglog])
with mlflow.start_run():
  mlflow.set_tag("modele", "tfidf-reglog")
  donnees = mlflow.data.from_spark(
    entrainement, path=options.donnees, name="entrainement"
  )
  mlflow.log_input(donnees, context="training")
  modele = pipeline.fit(entrainement)
  auc = BinaryClassificationEvaluator(labelCol="positif").evaluate(
    modele.transform(test)
  )
  mlflow.log_metric("auc_test", auc)
  mlflow.spark.log_model(modele, artifact_path="modele")
  print(f"AUC de test : {auc:.3f}")
""".lstrip()
)

## 5. Écrire le projet

Le projet `projet-avis` a son code d'entraînement ; la cellule d'exercice écrit son `MLproject`, avec le point d'entrée `main`, qui lance `entrainer.py` avec trois paramètres : `donnees` (un chemin, par défaut `drugsComTest_raw.tsv`), `reg` et `vocabulaire` (des nombres). Ajoutez-lui :

- `python_env.yaml`, son environnement : la version de Python de ce notebook (majeure et mineure, par exemple `3.12`), `pip` pour construire l'environnement, et les versions exactes de `mlflow` et `pyspark` que ce notebook utilise ;
- `evaluer.py`, un script qui reçoit l'URI d'un modèle (`runs:/<id>/modele` ou `models:/<nom>@<alias>`) et le fichier des avis, charge le modèle avec la saveur `spark`, calcule son AUC sur la partie test des avis (`lire_decouper` d'`avis.py`), la note en métrique `auc_test` et l'affiche ;
- dans le `MLproject`, un second point d'entrée, `evaluer`, à deux paramètres, `modele` (une URI, sans valeur par défaut) et `donnees` (comme celui de `main`), qui lance `evaluer.py` avec eux. L'exercice 7 l'appellera sur le modèle promu.

Pourquoi fixer les versions exactes des bibliothèques plutôt que les laisser libres ?


**Documentation utile**

- [MLflow Projects : environnement Python](https://mlflow.org/docs/latest/ml/projects/#python-virtual-environments-recommended)
- [MLflow Projects : types de paramètres](https://mlflow.org/docs/latest/ml/projects/#parameter-types)

In [ ]:
Path("projet-avis/MLproject").write_text(
  """
name: avis-medicaments

python_env: python_env.yaml

entry_points:
  main:
    parameters:
      donnees: {type: path, default: drugsComTest_raw.tsv}
      reg: {type: float, default: 0.1}
      vocabulaire: {type: float, default: 5000}
    command: "python entrainer.py {donnees} --reg {reg} --vocabulaire {vocabulaire}"
""".lstrip()
)
# Votre code ici

### Conseils supplémentaires

Écrivez les fichiers depuis Python, avec `Path(...).write_text(...)`, pour qu'« Exécuter tout » les recrée : `sys.version_info` donne la version de Python, `mlflow.__version__` et `pyspark.__version__` celles des bibliothèques. Les types de paramètres d'un `MLproject` sont `string`, `float`, `path` et `uri` ; un paramètre sans `default` est obligatoire, et MLflow remplace `{nom}` dans la commande par la valeur du paramètre. Dans `evaluer.py`, `mlflow.spark.load_model` charge le modèle d'une URI, et `mlflow.start_run()` rejoint l'exécution que `mlflow run` a créée, comme dans `entrainer.py`.

```python
version = f"{sys.version_info.major}.{sys.version_info.minor}"
Path("projet-avis/python_env.yaml").write_text(
  f"""
python: "{version}"
build_dependencies:
  - pip
dependencies:
  - mlflow=={mlflow.__version__}
  - ...
""".lstrip()
)
Path("projet-avis/evaluer.py").write_text(
  """
import argparse

import mlflow
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from avis import lire_decouper, session

arguments = argparse.ArgumentParser()
arguments.add_argument("modele")
arguments.add_argument("donnees")
options = arguments.parse_args()

spark = session("evaluer-avis")
_, test = lire_decouper(spark, options.donnees)
modele = ...
auc = ...
with mlflow.start_run():
  ...
print(f"AUC de test : {auc:.3f}")
""".lstrip()
)
Path("projet-avis/MLproject").write_text(
  """
...
  evaluer:
    parameters:
      modele: {type: uri}
      donnees: ...
    command: "python evaluer.py {modele} {donnees}"
""".lstrip()
)
```

### Solution

In [ ]:
version = f"{sys.version_info.major}.{sys.version_info.minor}"
Path("projet-avis/python_env.yaml").write_text(
  f"""
python: "{version}"
build_dependencies:
  - pip
dependencies:
  - mlflow=={mlflow.__version__}
  - pyspark=={pyspark.__version__}
""".lstrip()
)
Path("projet-avis/evaluer.py").write_text(
  """
import argparse

import mlflow
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from avis import lire_decouper, session

arguments = argparse.ArgumentParser()
arguments.add_argument("modele")
arguments.add_argument("donnees")
options = arguments.parse_args()

spark = session("evaluer-avis")
_, test = lire_decouper(spark, options.donnees)
modele = mlflow.spark.load_model(options.modele)
auc = BinaryClassificationEvaluator(labelCol="positif").evaluate(modele.transform(test))
with mlflow.start_run():
  mlflow.log_metric("auc_test", auc)
print(f"AUC de test : {auc:.3f}")
""".lstrip()
)
Path("projet-avis/MLproject").write_text(
  """
name: avis-medicaments

python_env: python_env.yaml

entry_points:
  main:
    parameters:
      donnees: {type: path, default: drugsComTest_raw.tsv}
      reg: {type: float, default: 0.1}
      vocabulaire: {type: float, default: 5000}
    command: "python entrainer.py {donnees} --reg {reg} --vocabulaire {vocabulaire}"
  evaluer:
    parameters:
      modele: {type: uri}
      donnees: {type: path, default: drugsComTest_raw.tsv}
    command: "python evaluer.py {modele} {donnees}"
""".lstrip()
)
print(Path("projet-avis/python_env.yaml").read_text())
# Des versions fixées rendent l'environnement reproductible : relancé dans six mois,
# le projet retrouvera les mêmes bibliothèques, et le même résultat. Sans version, pip
# prendrait les dernières, dont l'API ou les valeurs par défaut peuvent avoir changé
# (en MLflow 3.18, mlflow.spark.log_model prend encore artifact_path, quand d'autres
# saveurs sont passées à name). evaluer.py n'a pas besoin du découpage
# d'entraînement : il ne garde que test (_), le même que celui d'entrainer.py, puisque
# le découpage ne dépend que du texte des avis.

## 6. Lancer le projet

Lancez le projet avec `mlflow run`, deux fois : avec `reg` à 0,3, puis avec `reg` à 0,3 et `vocabulaire` à 2 000. Prenez l'environnement de ce notebook (`--env-manager local`) : construire celui de `python_env.yaml` prendrait une à deux minutes. Les exécutions vont dans l'expérience `avis-medicaments` : `mlflow.set_experiment` a mis son identifiant dans la variable d'environnement `MLFLOW_EXPERIMENT_ID`, dont héritent les commandes que lance le notebook.

Faites ensuite de `projet-avis` un dépôt git, avec un premier commit de ses cinq fichiers, et relancez-le avec `reg` à 0,3 et `vocabulaire` à 2 000.

Cherchez enfin les exécutions lancées comme projet (balise `mlflow.source.type` à `PROJECT`) et affichez leurs paramètres, leur AUC, leur point d'entrée et le commit git qu'elles ont noté. Que garantit le commit que ne garantissaient pas les exécutions des exercices 1 à 4 ? Les deux exécutions de mêmes paramètres ont-elles la même AUC ?

Pour aller plus loin : `--env-manager uv` construit l'environnement de `python_env.yaml` avec uv (une à deux minutes la première fois, puis réutilisé), et MLflow le note dans la commande qu'il affiche.


In [ ]:
# Votre code ici

### Conseils supplémentaires

Dans une cellule, une ligne qui commence par `!` est une commande du shell. `mlflow run <dossier>` prend les paramètres avec `-P nom=valeur` ; avec `MLFLOW_EXPERIMENT_ID` déjà posée, il refuse `--experiment-name`. Sur Colab, git veut un nom et une adresse pour commiter : `git -c user.name=... -c user.email=... commit`. Dans la recherche, les balises s'écrivent `tags.mlflow.source.type`, `tags.mlflow.project.entryPoint` et `tags.mlflow.source.git.commit`.

```python
!mlflow run projet-avis -P reg=0.3 --env-manager local
!mlflow run projet-avis -P reg=0.3 -P vocabulaire=2000 --env-manager local
!git -C projet-avis init -q
!git -C projet-avis add MLproject python_env.yaml avis.py entrainer.py evaluer.py
!git -C projet-avis -c user.name=Stagiaire -c user.email=stagiaire@example.com commit -q -m "Projet avis"
!mlflow run projet-avis ...
projets = mlflow.search_runs(
  experiment_names=["avis-medicaments"],
  filter_string="tags.mlflow.source.type = 'PROJECT'",
)
print(projets[[...]])
```

### Solution

In [ ]:
!mlflow run projet-avis -P reg=0.3 --env-manager local
!mlflow run projet-avis -P reg=0.3 -P vocabulaire=2000 --env-manager local
!git -C projet-avis init -q
!git -C projet-avis add MLproject python_env.yaml avis.py entrainer.py evaluer.py
!git -C projet-avis -c user.name=Stagiaire -c user.email=stagiaire@example.com commit -q -m "Projet avis"
!mlflow run projet-avis -P reg=0.3 -P vocabulaire=2000 --env-manager local
projets = mlflow.search_runs(
  experiment_names=["avis-medicaments"],
  filter_string="tags.mlflow.source.type = 'PROJECT'",
)
print(
  projets[
    [
      "params.reg",
      "params.vocabulaire",
      "metrics.auc_test",
      "tags.mlflow.project.entryPoint",
      "tags.mlflow.source.git.commit",
    ]
  ]
)
# Trois exécutions : la première a les paramètres de reg-0.3 de l'exercice 2, et la
# même AUC, 0,865 ; les deux autres, à 2 000 mots, 0,864. Seule la dernière a un
# commit (tags.mlflow.source.git.commit) : il désigne le code exact qui a produit le
# modèle, que git checkout retrouve, alors que les exécutions des exercices 1 à 4 ne
# gardent que le nom du programme qui les a lancées, dont le contenu a pu changer
# depuis. Les deux exécutions de mêmes paramètres ont la même AUC : le découpage ne
# dépend que du texte des avis, et l'entraînement que des données et des paramètres. À
# quelques millionièmes près seulement : l'ordre des calculs entre partitions peut
# varier d'un entraînement à l'autre.

## 7. Promouvoir le meilleur modèle

Prenez, parmi les exécutions lancées comme projet qui ont noté un commit git (les autres ne disent pas quel code a produit leur modèle), celle qui a la meilleure AUC (la plus récente, à égalité), et inscrivez son modèle dans le registre sous le nom `avis-medicaments`. Donnez à la version créée l'alias `champion`. Rechargez le modèle par son alias, avec la saveur `spark`, et calculez `prediction` des trois avis de `nouveaux_avis`. Lancez ensuite le point d'entrée `evaluer` du projet sur le modèle de l'alias : retrouvez-vous son AUC ?

En partant de l'alias, quelles informations retrouvez-vous jusqu'au code qui a produit le modèle ? Affichez-les.


**Documentation utile**

- [mlflow.register_model](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.register_model)
- [mlflow.client.MlflowClient.set_registered_model_alias](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.set_registered_model_alias)
- [mlflow.client.MlflowClient.get_model_version_by_alias](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html#mlflow.client.MlflowClient.get_model_version_by_alias)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`projets.dropna(subset=[...])` retire les exécutions sans commit ; triez les autres sur l'AUC puis sur `start_time`. `mlflow.register_model(uri, nom)` inscrit le modèle d'une URI `runs:/<id>/modele` et rend la version créée (attribut `version`). `MlflowClient().set_registered_model_alias(nom, alias, version)` pose l'alias, `get_model_version_by_alias(nom, alias)` le relit : la version a un `run_id`. L'URI `models:/<nom>@<alias>` désigne le modèle d'un alias ; `mlflow run` lance un autre point d'entrée que `main` avec `-e <nom>`.

```python
versionnees = projets.dropna(subset=["tags.mlflow.source.git.commit"])
meilleur = versionnees.sort_values([...], ascending=False).iloc[0]
version = mlflow.register_model(f"runs:/{meilleur.run_id}/modele", "avis-medicaments")
client = mlflow.MlflowClient()
client.set_registered_model_alias("avis-medicaments", "champion", version.version)
champion = mlflow.spark.load_model("models:/avis-medicaments@champion")
!mlflow run projet-avis -e evaluer -P modele=... --env-manager local
version_champion = client.get_model_version_by_alias(...)
execution = mlflow.get_run(version_champion.run_id)
print(execution.data.params, execution.data.tags[...])
```

### Solution

In [ ]:
versionnees = projets.dropna(subset=["tags.mlflow.source.git.commit"])
meilleur = versionnees.sort_values(
  ["metrics.auc_test", "start_time"], ascending=False
).iloc[0]
version = mlflow.register_model(f"runs:/{meilleur.run_id}/modele", "avis-medicaments")
client = mlflow.MlflowClient()
client.set_registered_model_alias("avis-medicaments", "champion", version.version)
champion = mlflow.spark.load_model("models:/avis-medicaments@champion")
champion.transform(spark.createDataFrame(nouveaux_avis)).select("prediction").show()
!mlflow run projet-avis -e evaluer -P modele=models:/avis-medicaments@champion --env-manager local
version_champion = client.get_model_version_by_alias("avis-medicaments", "champion")
execution = mlflow.get_run(version_champion.run_id)
print(version_champion.version, version_champion.run_id)
print(execution.data.params)
print(
  execution.data.tags["mlflow.source.name"],
  execution.data.tags["mlflow.source.git.commit"],
)
# Version 1 du modèle avis-medicaments, issue de la dernière exécution du projet, la
# seule versionnée (reg = 0,3, vocabulaire = 2 000, AUC de 0,864) : evaluer, lancé
# sur l'alias, retrouve la même AUC, puisqu'il recalcule le même test. De l'alias, on
# remonte à la version, de la version à son exécution (run_id), et de l'exécution à
# ses paramètres, au dossier du projet (mlflow.source.name) et au commit de son code :
# de quoi reproduire le modèle avec mlflow run, ces paramètres et ce commit. Les
# prédictions, 1, 0 et 1, sont celles du modèle de l'exercice 4, à 5 000 mots.